#### Integration of Apache Kafka and Spark Streaming

Let's give some (admittedly odd) context to this notebook. An alien entity has entered Portugal and suddenly starts turning citizens into scotsmen! Fear grips the country. We need to find out what is happening and how it is affecting the population.

[To understand what is going on](https://www.youtube.com/watch?v=qxDJMn-534Y)
(This is a Monty **Python** sketch)

We have been charged by the Portuguese government to analyze how the population has already changed. We will use **Spark Streaming** to stream data from our Kafka cluster and analyze live-streamed epidemiological data.

We will work with streams, stream-stream joins and stream-static joins.

Stream-stream joins involve combining two continuous flows of data, which is more complex because events from both sources may arrive at different times or out of order. To make this work, Spark must temporarily store the data from both streams to check for matches, within, for exampke, a 5-minute window. To prevent the system from running out of memory, you must define "watermarks" and time constraints, which tell the engine how long it needs to wait for late data before it can safely discard old records.

Stream-static joins are the simpler of the two join types, used primarily to enrich real-time data with fixed reference information. This allows you to add context to your streaming data without needing to manage complex state for the static side of the join.

<img src="img/scottish_portugal.png" height="500" width="700"/>

Structured Streaming treats a ``stream`` of data as a table that is updated in real time. An underlying process then regularly checks for updates and updates the table, if necessary. The API around Structured Streaming is designed in such a way that what works on your DataFrame, should also work on your streamed DataFrame! 

``Spark Streaming`` is a subset of Spark's functionalities that allows us to work with event-based data, as with our Kafka cluster. We set some global variables and import Schema Types to **structure our data**.

In [3]:
import pyspark.sql.functions as F

from pyspark.sql.types import StructType, StringType, DoubleType, StructField, IntegerType, TimestampType
from pyspark.sql import SparkSession

KAFKA_BOOTSTRAP_SERVERS = "localhost:8098"
KAFKA_TOPIC = "scotsmen"

We initialize a Spark Session. We import the ``Spark SQL Kafka Connector`` as a dependency. 

In [4]:
#Checking the installed Java version
!java -version
!pip install "pyspark==3.5.0" 
# Install Java 17
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless

!java -version

openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-124.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-124.04, mixed mode, sharing)
Hit:1 https://packages.cloud.google.com/apt cloud-sdk InRelease
Hit:2 https://cli.github.com/packages stable InRelease                         
Get:3 https://download.docker.com/linux/ubuntu noble InRelease [48.5 kB]       
Hit:4 https://archive.ubuntu.com/ubuntu noble InRelease                        
Get:5 https://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]       
Get:6 https://download.docker.com/linux/ubuntu noble/stable amd64 Packages [47.6 kB]
Hit:7 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Get:8 https://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]     
Get:9 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:10 https://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]    
Hit:11 https://us-east-1.ec

In [5]:
# Initialize local spark session
# Read from ``KAFKA_TOPIC``
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"


In [6]:
# Initialize local spark session
spark = SparkSession \
    .builder \
    .appName("kafka_streaming") \
    .config("spark.streaming.stopGracefullyOnShutdown", True) \
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0') \
    .config("spark.sql.shuffle.partitions", 4) \
    .master("local[*]") \
    .getOrCreate()

:: loading settings :: url = jar:file:/system/conda/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/zeus/.ivy2/cache
The jars for the packages stored in: /home/zeus/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f08c224c-c7f3-419a-bfe2-a6dc0a039c30;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.0 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 758ms :: artifacts dl 24ms
	:: mod

Using the ``subscribe-publish`` paradigm, we subscribe to the Kafka topic ``scotsmen``.

In [ ]:
streaming_df = spark.readStream.format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", KAFKA_TOPIC) \
    .option("startingOffsets", "earliest") \
    .load()    

In [7]:
# TODO: Instantiate the schema of the messages received
# Fields: district (String), new_scotsmen (Integer), timestamp (Timestamp)

scotsmen_schema =  StructType([
    StructField("district", StringType()),
    StructField("new_scotsmen", IntegerType()),
    StructField("timestamp", TimestampType())
])

In [8]:
json_df =  streaming_df.select(
    F.from_json(F.col("value").cast("string"), scotsmen_schema).alias("value"),
    "timestamp"
    )

In [9]:
# We instantiate an SQL view to inspect our data
json_df.select("value.*").createOrReplaceTempView("scotsmen")

In [10]:
# # Sample query from ``scotsmen`` table
scotsmen_query = spark.sql("SELECT * FROM scotsmen")

# query = scotsmen_query.writeStream.toTable("my_table")

Note that, as with the non-streaming API, there are ``transformations`` and ``actions``. Execution of a query operation on Spark Streaming is lazy.

### Input Sources & Sinks

Spark Structured Streaming supports different input sources and sinks. Supported input sources are:
1. Kafka Streams
2. Files on a distributed file system (HDFS, S3). Spark will read files from a directory
3. A Socket Source

While input sources specify the origin of the data, sinks specify where the data will be written. Those sinks can be:
1. Kafka sink: Pushes data to Kafka
2. Files sink: Writes the output to a file (JSON, parquet, CSV etc.)
3. ForEach sink: Can be used to for each row of a DataFrame for custom storage logic
4. Console sink: Used for testing
5. Memory: Used for debugging

``Memory`` and ``Console``sinks are very similar. ``Memory`` mode makes the data available in an in-memory table for interactive inspection.

In [11]:
# TODO: Write the results to the console
# Use outputMode "append" and format "console"

query = scotsmen_query.writeStream.outputMode("append").format("console").start()

25/12/03 09:19:45 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-2e02c383-4a46-40ff-9772-89bebaf89316. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/12/03 09:19:45 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [12]:
# Stopping the query
query.stop()

There are three different output modes available. Here, we used ``append``, which only adds new records to the sink. The other two are ``update`` and ``complete``. ``update`` mode updates the data in the sink, while ``complete`` mode replaces the data in the sink.

In [13]:
# TODO: Write this query to a memory table called "scotsmen_table"
# Use outputMode "append" and format "memory"

memory_query = scotsmen_query.writeStream.outputMode("append").queryName("scotsmen_table").format("memory").start()

25/12/03 09:22:41 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-cb6a0486-c32b-41fb-878a-80163d1dcd22. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/12/03 09:22:41 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


25/12/03 09:22:42 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


In [14]:
spark.sql("SELECT district, sum(new_scotsmen) AS total_new_scotsmen FROM scotsmen_table GROUP BY district").show()

+--------------------+------------------+
|            district|total_new_scotsmen|
+--------------------+------------------+
|          Portalegre|             14328|
|             Setubal|              9958|
|    Viana do Castelo|             10515|
|      Castelo Branco|             14263|
|            Braganca|              9039|
|Regiao Autonoma d...|              7888|
|              Lisboa|              9343|
|               Braga|              3775|
|            Santarem|              9993|
|               Viseu|             17236|
|               Porto|              7454|
|           Vila Real|              6906|
|Regiao Autonoma d...|             10836|
|              Aveiro|             11379|
|                Faro|             11782|
|                Beja|             12630|
|               Evora|              9878|
|              Guarda|             11094|
|              Leiria|              8174|
|             Coimbra|              3617|
+--------------------+------------

#### Window Functions

In [15]:
# Convert JSON to DataFrame
new_scotsmen_df = json_df.select(
    F.col("value.district").alias("conversion_district"),
    F.col("value.timestamp").alias("conversion_timestamp"),
    F.col("value.new_scotsmen").alias("new_scotsmen")
)

In [16]:
# TODO: Apply Watermarking and Windowing
# 1. Add watermark (30 seconds) on "conversion_timestamp"
# 2. Group by a 3 minute window on "conversion_timestamp" AND "conversion_district"
# 3. Aggregate: count, sum(new_scotsmen), and avg(new_scotsmen)
windowed_df = new_scotsmen_df.withWatermark("conversion_timestamp", "30 seconds") \
    .groupBy(
        F.window(new_scotsmen_df["conversion_timestamp"], "3 minute"), 
        new_scotsmen_df["conversion_district"]
    ) \
    .agg(
        F.count("*").alias("event_count"),
        F.sum("new_scotsmen").alias("total_new_scotsmen"), # sum of values for each district in each window
        F.avg("new_scotsmen").alias("average_new_scotsmen") # compute avg value for each diostrict in each window
    )

In [17]:
window_query = windowed_df \
    .writeStream \
    .outputMode("complete") \
    .queryName("new_scotsmen_aggregated") \
    .format("memory") \
    .start()

25/12/03 09:28:16 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-7f7b71f6-c800-471e-9967-daee3b61aa6c. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/12/03 09:28:16 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


25/12/03 09:28:16 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


In [18]:
spark.sql("SELECT * FROM new_scotsmen_aggregated").show()

+--------------------+--------------------+-----------+------------------+--------------------+
|              window| conversion_district|event_count|total_new_scotsmen|average_new_scotsmen|
+--------------------+--------------------+-----------+------------------+--------------------+
|{2025-12-03 09:18...|              Guarda|          5|              5100|              1020.0|
|{2025-12-03 09:09...|              Lisboa|          2|              1284|               642.0|
|{2025-12-03 09:21...|Regiao Autonoma d...|          2|              2094|              1047.0|
|{2025-12-03 09:12...|             Coimbra|          3|              2091|               697.0|
|{2025-12-03 09:00...|           Vila Real|          2|              1576|               788.0|
|{2025-12-03 09:00...|               Braga|          1|              1299|              1299.0|
|{2025-12-03 09:18...|              Leiria|          2|               700|               350.0|
|{2025-12-03 09:09...|             Setub

In [19]:
spark.sql("""
    SELECT window.start, window.end, conversion_district, total_new_scotsmen 
    FROM new_scotsmen_aggregated 
    ORDER BY window.start DESC
""").show(truncate=False)

+-------------------+-------------------+--------------------------+------------------+
|start              |end                |conversion_district       |total_new_scotsmen|
+-------------------+-------------------+--------------------------+------------------+
|2025-12-03 09:30:00|2025-12-03 09:33:00|Viana do Castelo          |1492              |
|2025-12-03 09:30:00|2025-12-03 09:33:00|Coimbra                   |712               |
|2025-12-03 09:30:00|2025-12-03 09:33:00|Vila Real                 |905               |
|2025-12-03 09:30:00|2025-12-03 09:33:00|Viseu                     |1096              |
|2025-12-03 09:30:00|2025-12-03 09:33:00|Castelo Branco            |371               |
|2025-12-03 09:30:00|2025-12-03 09:33:00|Guarda                    |1951              |
|2025-12-03 09:30:00|2025-12-03 09:33:00|Porto                     |3627              |
|2025-12-03 09:30:00|2025-12-03 09:33:00|Regiao Autonoma da Madeira|1097              |
|2025-12-03 09:27:00|2025-12-03 

In [20]:
spark.sql("""
    SELECT conversion_district, total_new_scotsmen, window.start 
    FROM new_scotsmen_aggregated 
    ORDER BY total_new_scotsmen DESC 
    LIMIT 5
""").show()

+-------------------+------------------+-------------------+
|conversion_district|total_new_scotsmen|              start|
+-------------------+------------------+-------------------+
|              Viseu|              5590|2025-12-03 09:12:00|
|             Guarda|              5100|2025-12-03 09:18:00|
|              Viseu|              4925|2025-12-03 09:06:00|
|              Porto|              4904|2025-12-03 09:30:00|
|               Faro|              4309|2025-12-03 09:18:00|
+-------------------+------------------+-------------------+



#### Advanced Features

Structured Streaming supports ``Joins``. This means that you are able to (I) join a stream with a static DataFrame and (II) join two streams. This can be used to supplement streaming data with another data source.

Here, we will supplement our ``scotsmen`` table with the ``bag_pipes_sales`` table.

In [21]:
# Again, we need to read from Kafka.
# This time, we subscribe to the bagpipes topic
bagpipes_stream = spark.readStream.format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", "bagpipe") \
    .option("startingOffsets", "earliest") \
    .load()    

In [22]:
# Instantiate the schema of the messages received
bagpipes_schema = StructType([
  StructField("district", StringType()),
  StructField("bagpipe_sales", IntegerType()),
  StructField("timestamp", TimestampType())
])

In [23]:
# We select the 'value' column and cast it as a String
bagpipes_json_df = bagpipes_stream.select(
    F.from_json(F.col("value").cast("string"), bagpipes_schema).alias("value"), 
    )

In [24]:
# Convert the JSON to DataFrame and alias columns
bagpipe_df = bagpipes_json_df.select(
    F.col("value.district").alias("sales_district"),
    F.col("value.timestamp").alias("sale_timestamp"),
    F.col("value.bagpipe_sales").alias("bagpipe_sales")
)

In [29]:
# Add watermarking to both streams to handle late data
conversion_stream = new_scotsmen_df.withColumn("conversion_truncated_timestamp", F.date_trunc("minute", new_scotsmen_df["conversion_timestamp"]))
bagpipes_stream = bagpipe_df.withColumn("sales_truncated_timestamp", F.date_trunc("minute", bagpipe_df["sale_timestamp"]))

# Watermark the datasets
conversion_stream = conversion_stream.withWatermark("conversion_truncated_timestamp", "1 minute")
bagpipes_stream = bagpipes_stream.withWatermark("sales_truncated_timestamp", "1 minute")

# Alias the datasets
conversion_stream = conversion_stream.alias("s1")
bagpipes_stream = bagpipes_stream.alias("s2")

# Perform the join between the two windowed streams on 'district' and matching 
# windows by using a functional expression
joined_stream = conversion_stream \
    .join(
        bagpipes_stream,
        F.expr("""
            s1.conversion_district = s2.sales_district AND
            s2.sales_truncated_timestamp >= s1.conversion_truncated_timestamp AND
            s2.sales_truncated_timestamp <= s1.conversion_truncated_timestamp + interval 5 minute
        """)
    ) \
    .select(
        "s2.sales_truncated_timestamp",
        "s1.new_scotsmen",
        "s2.bagpipe_sales",
        "s1.conversion_district"
    )

# Create the window column before aggregation
joined_stream = joined_stream.withColumn("window", F.window("sales_truncated_timestamp", "2 minutes"))

# Aggregate values per district
aggregated_stream = joined_stream \
    .groupBy(
        joined_stream.conversion_district,
        joined_stream.window
    ) \
    .agg(
        F.sum(joined_stream.new_scotsmen).alias("total_new_scotsmen"),
        F.sum(joined_stream.bagpipe_sales).alias("total_bagpipe_sales")
    )

In [ ]:
# Output the results to the console for inspection
query = aggregated_stream \
    .writeStream \
    .queryName("stream_stream_join") \
    .outputMode("append") \
    .format("memory") \
    .start()

25/12/03 09:38:01 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-c2371928-6597-477d-8e54-4a707b051e25. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/12/03 09:38:01 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
25/12/03 09:38:02 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


25/12/03 09:38:02 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


In [28]:
query.stop()

In [31]:
# Showing the results
spark.sql("SELECT * FROM stream_stream_join").show()

+-------------------+------+------------------+-------------------+
|conversion_district|window|total_new_scotsmen|total_bagpipe_sales|
+-------------------+------+------------------+-------------------+
+-------------------+------+------------------+-------------------+



#### Static-Stream Joins

Apart from joining two streams, Spark also supports joining a stream with a static DataFrame. This can be used to supplement streaming data with another data source, such as a lookup table. Here  we will supplement our ``conversation_stream`` with the ``portugal_district_population2022.csv`` table.

In [32]:
population_schema = StructType([
  StructField("district", StringType()),
  StructField("pop", IntegerType())
])

population_df = spark \
    .read \
    .format("csv") \
    .option("header", True) \
    .schema(population_schema) \
    .load("portugal_district_population2022.csv")

In [33]:
# TODO: Join the conversion_stream with the static population_df
# Match on district columns
joined_stream_population = conversion_stream.join(
    population_df, 
    conversion_stream.conversion_district == population_df.district, 
    "inner"
)#.withColumn("conversion_per_pop", F.col("new_scotsmen") / F.col("pop"))

# Calculate conversions per population
joined_stream_population = joined_stream_population \
    .withColumn(
        "conversions_per_pop", F.col("new_scotsmen") / F.col("pop")
    )

# Select the columns you need
result_stream = joined_stream_population.select(
    "conversion_truncated_timestamp",
    "new_scotsmen",
    "conversion_district",
    "pop",
    "conversions_per_pop"
)

In [34]:
# Output the results to the console for testing
query = result_stream \
    .writeStream \
    .queryName("prop_pop_converted") \
    .outputMode("append") \
    .format("memory") \
    .start()

25/12/03 09:40:38 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-de8627bb-a986-4e8f-be85-2d48a67cf3ed. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/12/03 09:40:38 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


25/12/03 09:40:39 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


In [37]:
# Define the window duration and slide duration
window_duration = "1 hour"
slide_duration = "10 minutes"

# SQL query to compute the cumulative sum of the ratio
sql_query = f"""
SELECT
    window.start AS window_start,
    window.end AS window_end,
    SUM(conversions_per_pop) AS cumulative_ratio,
    conversion_district
FROM (
    SELECT
        conversions_per_pop,
        window(current_timestamp(), '{window_duration}', '{slide_duration}') AS window,
        conversion_district
    FROM prop_pop_converted
)
GROUP BY conversion_district, window
ORDER BY window_start
"""

# Execute the SQL query
result_df = spark.sql(sql_query)

# Show the result
result_df.show()

+-------------------+-------------------+--------------------+--------------------+
|       window_start|         window_end|    cumulative_ratio| conversion_district|
+-------------------+-------------------+--------------------+--------------------+
|2025-12-03 08:50:00|2025-12-03 09:50:00|  0.0751657275439178|               Viseu|
|2025-12-03 08:50:00|2025-12-03 09:50:00| 0.09283868745155456|    Viana do Castelo|
|2025-12-03 08:50:00|2025-12-03 09:50:00| 0.14353558145161743|      Castelo Branco|
|2025-12-03 08:50:00|2025-12-03 09:50:00| 0.08493676434006295|Regiao Autonoma d...|
|2025-12-03 08:50:00|2025-12-03 09:50:00| 0.10335420305783989|               Evora|
|2025-12-03 08:50:00|2025-12-03 09:50:00| 0.10376537048442246|              Guarda|
|2025-12-03 08:50:00|2025-12-03 09:50:00| 0.23799233584003227|          Portalegre|
|2025-12-03 08:50:00|2025-12-03 09:50:00| 0.01646192033588309|             Setubal|
|2025-12-03 08:50:00|2025-12-03 09:50:00|  0.1736718350655989|              

#### Simulating Streaming Datasets

It is also possible to "simulate" a streaming dataset by reading from a directory of CSV files. This can be useful for testing and debugging purposes. The dataset contains individual files that can be read as a batch of data.

In [7]:
# Define the weather schema
schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("min_temperature", DoubleType(), True),
    StructField("max_temperature", DoubleType(), True),
    StructField("precipitation", DoubleType(), True)
])

# Path to the directory containing the CSV files
# Note that this must be a directory and not an individual file
input_path = "weather"

NUM_FILES_PER_TRIGGER = 3

# Read the streaming DataFrame from the directory
streaming_df = spark.readStream \
    .option("maxFilesPerTrigger", NUM_FILES_PER_TRIGGER) \
    .option("header", "true") \
    .format("csv") \
    .schema(schema) \
    .load(input_path)

# Define the query to process the streaming data
# It is possible to set the batch size 
# to control how frequently the streaming query processes new data.
# This is done using the trigger option in the writeStream method. 
# The trigger option allows you to specify the processing time interval, 
# which determines the batch size.
# maxFilesPerTrigger is the maximum number of files that will be
# processed in a single trigger.
query = streaming_df.writeStream \
    .trigger(processingTime='10 seconds') \
    .option("maxFilesPerTrigger", 5) \
    .outputMode("append") \
    .queryName("weather") \
    .format("memory") \
    .start()

25/12/03 15:04:17 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-7ada962f-0ccb-452c-aa8d-2592476b44be. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/12/03 15:04:17 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [9]:
# We can now query the static dataset just like we did before
spark.sql(
    """SELECT max_temperature, min_temperature, timestamp 
    FROM weather ORDER BY timestamp 
    DESC
    """
).show()

+------------------+------------------+-------------------+
|   max_temperature|   min_temperature|          timestamp|
+------------------+------------------+-------------------+
| 35.71719088731736| 28.92753570022043|2024-01-03 00:20:00|
| 34.29180514598012|26.291868805945914|2024-01-03 00:19:00|
| 24.88623899422729|15.012854166069987|2024-01-03 00:18:00|
| 21.17516218891879|15.213430893058344|2024-01-03 00:17:00|
|30.679984357788623| 18.29915145603222|2024-01-03 00:16:00|
| 43.01813017620789|29.307721665233363|2024-01-03 00:15:00|
| 22.63812559275722| 11.01065774058517|2024-01-03 00:14:00|
|24.133548867471113| 17.43335801908966|2024-01-03 00:13:00|
| 38.66024356767653|29.534416561153854|2024-01-03 00:12:00|
|41.114689804217015|29.920882899824676|2024-01-03 00:11:00|
|22.030487680422574| 15.69629571938437|2024-01-03 00:10:00|
|24.466284554938152|17.740716870098332|2024-01-03 00:09:00|
|32.178807707662514|26.332308441840357|2024-01-03 00:08:00|
| 33.20548152488369|20.800442727142556|2

### 1. Cleaning and enriching the simulated weather stream

We now treat the `streaming_df` as our raw input stream and build a small ETL layer on top of it.

We will:
- Drop physically impossible values for temperature.
- Add calendar features (day of week, hour of day).
- Add a simple engineering feature: temperature range.

We then write the cleaned stream to the console so we can see the events as they arrive.


In [10]:
from pyspark.sql import functions as F

# Clean and enrich streaming weather data
weather_clean = (
    streaming_df
        # Drop impossible min temperatures
        .filter(
            (F.col("min_temperature") >= -40) &
            (F.col("min_temperature") <= 60)
        )
        # Optionally restrict max_temperature as well
        # .filter((F.col("max_temperature") >= -40) & (F.col("max_temperature") <= 60))
        .withColumn("day_of_week", F.date_format("timestamp", "E"))  # Mon, Tue, ...
        .withColumn("hour", F.hour("timestamp"))                     # 0–23
        .withColumn(
            "temperature_range",
            F.col("max_temperature") - F.col("min_temperature")
        )
)

weather_clean_query = (
    weather_clean
        .writeStream
        .format("console")
        .outputMode("append")
        .option("truncate", "false")
        .start()
)


25/12/03 16:29:09 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-62487029-aaf6-49d0-9f6a-729b88dd8b30. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/12/03 16:29:09 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+------------------+------------------+------------------+-----------+----+------------------+
|timestamp          |min_temperature   |max_temperature   |precipitation     |day_of_week|hour|temperature_range |
+-------------------+------------------+------------------+------------------+-----------+----+------------------+
|2024-01-03 00:01:00|29.555378740901755|35.38688199302494 |15.102537333979981|Wed        |0   |5.831503252123188 |
|2024-01-03 00:02:00|16.26954231302708 |23.338552855877396|25.396788595236202|Wed        |0   |7.0690105428503145|
|2024-01-03 00:00:00|26.441340905014673|39.5395708121858  |47.85778956217611 |Wed        |0   |13.098229907171127|
+-------------------+------------------+------------------+------------------+-----------+----+------------------+

-------------------------------------------
Batch: 1
-------------------------------------------


In [11]:
weather_clean_query.stop()


### 2. Event-time windowing and watermarking

Next, we use event-time windows and watermarks on the `timestamp` column.

We:

* Define a watermark of 30 minutes, which tells Spark how long to wait for late data.
* Use 30-minute windows with a 10-minute slide.
* Compute aggregate statistics per window.

We write the results to an in-memory table so that we can query it with SQL.

In [12]:

windowed_weather = (
    weather_clean
        .withWatermark("timestamp", "30 minutes")
        .groupBy(
            F.window("timestamp", "30 minutes", "10 minutes")
        )
        .agg(
            F.avg("max_temperature").alias("avg_max_temp"),
            F.avg("min_temperature").alias("avg_min_temp"),
            F.sum("precipitation").alias("total_precip"),
            F.avg("temperature_range").alias("avg_temp_range")
        )
)

weather_windows_query = (
    windowed_weather
        .writeStream
        .format("memory")
        .outputMode("update")
        .queryName("weather_windows")
        .option("checkpointLocation", "checkpoints/weather_windows")
        .start()
)

25/12/03 16:34:55 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [13]:
spark.sql("""
    SELECT
        window.start AS window_start,
        window.end   AS window_end,
        avg_max_temp,
        avg_min_temp,
        total_precip,
        avg_temp_range
    FROM weather_windows
    ORDER BY window_start
""").show(truncate=False)

+------------+----------+------------+------------+------------+--------------+
|window_start|window_end|avg_max_temp|avg_min_temp|total_precip|avg_temp_range|
+------------+----------+------------+------------+------------+--------------+
+------------+----------+------------+------------+------------+--------------+



When you are done inspecting the windowed stream, you can stop it with:

In [14]:
weather_windows_query.stop()



### 3. Streaming ETL with `foreachBatch`: writing curated Parquet

Now we show a realistic ETL pattern in Structured Streaming.

We:

* Use `foreachBatch` with a Python function that is called once per micro-batch.
* Inside this function, we use normal Spark DataFrame writes to append the batch to Parquet.

This is appropriate for big data because:

* The `write_batch_to_parquet` function is only orchestration on the driver.
* The `.write.parquet(...)` call is executed in parallel on executors, just like any other Spark write.
* We avoid doing I/O inside UDFs or pandas UDFs, which would create many small, uncoordinated writes and break fault tolerance.

UDFs and pandas UDFs are meant for transforming columns inside a DataFrame, not for side effects such as writing micro-batches to storage.


In [15]:

output_path = "curated_weather_parquet"

def write_batch_to_parquet(batch_df, batch_id):
    # This function is called once per micro-batch on the driver.
    # The write operation itself is distributed across the cluster.
    (
        batch_df
            .select(
                "timestamp",
                "min_temperature",
                "max_temperature",
                "precipitation",
                "temperature_range",
                "day_of_week",
                "hour"
            )
            .write
            .mode("append")
            .parquet(output_path)
    )

etl_query = (
    weather_clean
        .writeStream
        .foreachBatch(write_batch_to_parquet)
        .outputMode("append")
        .option("checkpointLocation", "checkpoints/weather_etl")
        .start()
)

25/12/03 16:34:57 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [ ]:

25/12/04 10:42:28 WARN StreamingQueryManager: Stopping existing streaming query [id=2dd816d4-1a5d-4d96-b531-5d75b2801682, runId=ce50a90e-a03e-46f6-9c55-40ca425845e7], as a new run is being started.

We can now read the curated Parquet data back as a batch DataFrame and compute some simple statistics.

In [17]:

curated = spark.read.parquet("/teamspace/studios/this_studio/notebooks_solved/week13/curated_weather_parquet")

curated.groupBy("day_of_week").agg(
    F.avg("max_temperature").alias("avg_max_temp"),
    F.avg("min_temperature").alias("avg_min_temp")
).orderBy("day_of_week").show()

+-----------+------------------+------------------+
|day_of_week|      avg_max_temp|      avg_min_temp|
+-----------+------------------+------------------+
|        Wed|29.530361122790303|19.479908167304632|
+-----------+------------------+------------------+



When you are done with the ETL stream, you can stop it with:

In [18]:
etl_query.stop()

### 4. Batch training a RandomForest model on historical weather

We now build a non-trivial MLlib pipeline using RandomForest.

Steps:

* Read the same weather folder in batch mode.
* Apply the same cleaning and feature engineering logic used for the stream.
* Define a label `is_rainy` (1 if `precipitation > 0`, 0 otherwise).
* Encode `day_of_week` as a numeric feature.
* Assemble features and train a RandomForest classifier.

In [19]:

from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

# Read historical weather data as batch
weather_history = (
    spark.read
        .format("csv")
        .schema(schema)  # reuse the same schema defined earlier
        .option("header", "true")
        .load("weather")
        .dropna(subset=["timestamp", "min_temperature", "max_temperature", "precipitation"])
)

# Reuse the same cleaning / feature engineering logic
weather_history = (
    weather_history
        .filter(
            (F.col("min_temperature") >= -40) &
            (F.col("min_temperature") <= 60)
        )
        .withColumn("day_of_week", F.date_format("timestamp", "E"))
        .withColumn("hour", F.hour("timestamp"))
        .withColumn(
            "temperature_range",
            F.col("max_temperature") - F.col("min_temperature")
        )
)

# Classification label: rainy or not
training_df = weather_history.withColumn(
    "is_rainy",
    (F.col("precipitation") > 0).cast("int")
)

# Index day_of_week into a numeric feature
day_indexer = StringIndexer(
    inputCol="day_of_week",
    outputCol="day_index",
    handleInvalid="keep"
)

feature_cols = [
    "min_temperature",
    "max_temperature",
    "temperature_range",
    "hour",
    "day_index"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="is_rainy",
    numTrees=50,
    maxDepth=8,
    seed=42
)

rf_pipeline = Pipeline(stages=[day_indexer, assembler, rf])

rain_model = rf_pipeline.fit(training_df)

# Optional: save the model to reuse without retraining
# rain_model.write().overwrite().save("models/rain_random_forest")

We now have a trained RandomForest model that can classify each record as rainy or not, based on the weather features.


### 5. Applying the RandomForest model to the weather stream

We apply the trained model to the streaming DataFrame `weather_clean`.

This produces a new streaming DataFrame with:
- The original weather columns.
- The prediction (0 or 1).
- The probability vector from RandomForest.

In [20]:

# If you saved the model earlier, you could load it instead:
# from pyspark.ml import PipelineModel
# rain_model = PipelineModel.load("models/rain_random_forest")

predictions_stream = rain_model.transform(weather_clean)

predictions_query = (
    predictions_stream
        .select(
            "timestamp",
            "min_temperature",
            "max_temperature",
            "precipitation",
            "temperature_range",
            "day_of_week",
            "hour",
            "probability",
            "prediction"    # 1 = rainy, 0 = not rainy
        )
        .writeStream
        .format("console")
        .outputMode("append")
        .option("truncate", "false")
        .start()
)

25/12/03 16:36:13 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-74a6b4b2-bcc6-4370-99ea-46568cb98ae7. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/12/03 16:36:13 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+------------------+------------------+------------------+------------------+-----------+----+-----------+----------+
|timestamp          |min_temperature   |max_temperature   |precipitation     |temperature_range |day_of_week|hour|probability|prediction|
+-------------------+------------------+------------------+------------------+------------------+-----------+----+-----------+----------+
|2024-01-03 00:01:00|29.555378740901755|35.38688199302494 |15.102537333979981|5.831503252123188 |Wed        |0   |[0.0,1.0]  |1.0       |
|2024-01-03 00:02:00|16.26954231302708 |23.338552855877396|25.396788595236202|7.0690105428503145|Wed        |0   |[0.0,1.0]  |1.0       |
|2024-01-03 00:00:00|26.441340905014673|39.5395708121858  |47.85778956217611 |13.098229907171127|Wed        |0   |[0.0,1.0]  |1.0       |
+-------------------+------------------+------------------+----------------


You can inspect the predictions in the console. When finished, stop the query:


In [21]:
predictions_query.stop()


-------------------------------------------
Batch: 26
-------------------------------------------
+-------------------+------------------+-----------------+-----------------+------------------+-----------+----+-----------+----------+
|timestamp          |min_temperature   |max_temperature  |precipitation    |temperature_range |day_of_week|hour|probability|prediction|
+-------------------+------------------+-----------------+-----------------+------------------+-----------+----+-----------+----------+
|2024-01-03 01:18:00|11.700488004361432|21.55863306318198|42.70880804335865|9.858145058820547 |Wed        |1   |[0.0,1.0]  |1.0       |
|2024-01-03 01:20:00|29.121189526835998|37.60411676847761|42.87763569629343|8.482927241641612 |Wed        |1   |[0.0,1.0]  |1.0       |
|2024-01-03 01:19:00|16.16352324737735 |28.07655947364174|16.42650332495282|11.913036226264392|Wed        |1   |[0.0,1.0]  |1.0       |
+-------------------+------------------+-----------------+-----------------+----------


### 6. Adding a rain risk label and filtering high-risk predictions

We now:

* Add a human-readable `rain_risk` column that maps prediction to "High" or "Low".
* Filter the stream to show only high-risk rows.


In [22]:
# Add a risk label based on the prediction
predictions_enriched = (
    predictions_stream
        .withColumn(
            "rain_risk",
            F.when(F.col("prediction") == 1, F.lit("High"))
             .otherwise(F.lit("Low"))
        )
)

rain_risk_query = (
    predictions_enriched
        .select(
            "timestamp",
            "min_temperature",
            "max_temperature",
            "precipitation",
            "temperature_range",
            "day_of_week",
            "hour",
            "prediction",
            "rain_risk"
        )
        .writeStream
        .format("console")
        .outputMode("append")
        .option("truncate", "false")
        .start()
)

25/12/03 16:36:26 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-6bc5f611-cc52-45a1-8d67-a456375e619d. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/12/03 16:36:26 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+------------------+------------------+------------------+------------------+-----------+----+----------+---------+
|timestamp          |min_temperature   |max_temperature   |precipitation     |temperature_range |day_of_week|hour|prediction|rain_risk|
+-------------------+------------------+------------------+------------------+------------------+-----------+----+----------+---------+
|2024-01-03 00:01:00|29.555378740901755|35.38688199302494 |15.102537333979981|5.831503252123188 |Wed        |0   |1.0       |High     |
|2024-01-03 00:02:00|16.26954231302708 |23.338552855877396|25.396788595236202|7.0690105428503145|Wed        |0   |1.0       |High     |
|2024-01-03 00:00:00|26.441340905014673|39.5395708121858  |47.85778956217611 |13.098229907171127|Wed        |0   |1.0       |High     |
+-------------------+------------------+------------------+------------------+---------

To see only high-risk predictions, we filter the streaming DataFrame before writing.

In [23]:

high_risk_stream = predictions_enriched.filter(F.col("rain_risk") == "High")

high_risk_query = (
    high_risk_stream
        .select(
            "timestamp",
            "min_temperature",
            "max_temperature",
            "precipitation",
            "temperature_range",
            "day_of_week",
            "hour",
            "rain_risk"
        )
        .writeStream
        .format("console")
        .outputMode("append")
        .option("truncate", "false")
        .start()
)

-------------------------------------------
Batch: 21
-------------------------------------------
+-------------------+------------------+------------------+------------------+-----------------+-----------+----+----------+---------+
|timestamp          |min_temperature   |max_temperature   |precipitation     |temperature_range|day_of_week|hour|prediction|rain_risk|
+-------------------+------------------+------------------+------------------+-----------------+-----------+----+----------+---------+
|2024-01-03 01:05:00|12.107133773290732|19.106250419861883|1.1188024303647681|6.999116646571151|Wed        |1   |1.0       |High     |
|2024-01-03 01:03:00|14.121556968371445|20.437538902606942|22.75249649564423 |6.315981934235497|Wed        |1   |1.0       |High     |
|2024-01-03 01:04:00|26.078336834570713|34.930643857670354|5.638945934249012 |8.852307023099641|Wed        |1   |1.0       |High     |
+-------------------+------------------+------------------+------------------+--------------

25/12/03 16:36:32 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-ed68404d-9c84-4b7e-8d7c-65622bc74903. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/12/03 16:36:32 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 22
-------------------------------------------
+-------------------+------------------+------------------+------------------+------------------+-----------+----+----------+---------+
|timestamp          |min_temperature   |max_temperature   |precipitation     |temperature_range |day_of_week|hour|prediction|rain_risk|
+-------------------+------------------+------------------+------------------+------------------+-----------+----+----------+---------+
|2024-01-03 01:07:00|12.623302780953287|23.37855192039968 |13.161695006948554|10.755249139446393|Wed        |1   |1.0       |High     |
|2024-01-03 01:08:00|14.24910192995068 |24.321551779075044|16.982659134484123|10.072449849124364|Wed        |1   |1.0       |High     |
|2024-01-03 01:06:00|11.165328346955928|22.40091566759773 |43.37183486134943 |11.235587320641802|Wed        |1   |1.0       |High     |
+-------------------+------------------+------------------+------------------+--------

You can stop these queries with:


In [24]:


rain_risk_query.stop()
high_risk_query.stop()

25/12/03 16:36:39 WARN Shell: Interrupted while joining on: Thread[Thread-10584,5,main]
java.lang.InterruptedException
	at java.base/java.lang.Object.wait(Native Method)
	at java.base/java.lang.Thread.join(Thread.java:1313)
	at java.base/java.lang.Thread.join(Thread.java:1381)
	at org.apache.hadoop.util.Shell.joinThread(Shell.java:1042)
	at org.apache.hadoop.util.Shell.runCommand(Shell.java:1002)
	at org.apache.hadoop.util.Shell.run(Shell.java:900)
	at org.apache.hadoop.util.Shell$ShellCommandExecutor.execute(Shell.java:1212)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1306)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1288)
	at org.apache.hadoop.fs.FileUtil.readLink(FileUtil.java:212)
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileLinkStatusInternal(RawLocalFileSystem.java:1113)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1102)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatus(RawLocalFi

-------------------------------------------
Batch: 14
-------------------------------------------
+-------------------+------------------+------------------+------------------+------------------+-----------+----+---------+
|timestamp          |min_temperature   |max_temperature   |precipitation     |temperature_range |day_of_week|hour|rain_risk|
+-------------------+------------------+------------------+------------------+------------------+-----------+----+---------+
|2024-01-03 00:44:00|27.379093206422898|41.258135959216894|20.884194962571666|13.879042752793996|Wed        |0   |High     |
|2024-01-03 00:42:00|10.53070570561951 |18.326461019951193|40.658789516957384|7.795755314331682 |Wed        |0   |High     |
|2024-01-03 00:43:00|26.975392175629388|37.60762301884307 |18.375716125703917|10.632230843213684|Wed        |0   |High     |
+-------------------+------------------+------------------+------------------+------------------+-----------+----+---------+

